In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv('../outputs/predictions.csv')

# Month cohort
np.random.seed(42)
df['month_cohort'] = np.random.choice(
    ['Month 1','Month 2','Month 3','Month 4','Month 5','Month 6'],
    size=len(df),
    p=[0.10, 0.15, 0.18, 0.20, 0.22, 0.15]
)

# Month cohort order number
month_order = {
    'Month 1': 1, 'Month 2': 2, 'Month 3': 3,
    'Month 4': 4, 'Month 5': 5, 'Month 6': 6
}
df['month_cohort_num'] = df['month_cohort'].map(month_order)

# Revenue at risk
df['annual_revenue_at_risk'] = df['MonthlyCharges'] * 12 * df['churn_probability']

# Probability bucket
df['prob_bucket'] = pd.cut(
    df['churn_probability'],
    bins=[0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
    labels=['0-10%','10-20%','20-30%','30-40%','40-50%',
            '50-60%','60-70%','70-80%','80-90%','90-100%']
)

# Tenure group label
def tenure_label(t):
    if t <= 12:   return '0-12 months'
    elif t <= 24: return '13-24 months'
    elif t <= 48: return '25-48 months'
    else:         return '49+ months'

df['tenure_group_label'] = df['tenure'].apply(tenure_label)

# Risk score
df['risk_score'] = (df['churn_probability'] * 100).round(1)

# Prediction label
df['prediction_correct'] = (df['predicted_churn'] == df['actual_churn']).astype(int)
df['prediction_label'] = df.apply(
    lambda r: 'True Positive'  if r['predicted_churn']==1 and r['actual_churn']==1
    else ('True Negative'      if r['predicted_churn']==0 and r['actual_churn']==0
    else ('False Positive'     if r['predicted_churn']==1 and r['actual_churn']==0
    else 'False Negative')), axis=1
)

df.to_csv('../outputs/predictions_enhanced.csv', index=False)
print("Saved: outputs/predictions_enhanced.csv")
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

Saved: outputs/predictions_enhanced.csv
Shape: (1409, 16)
Columns: ['churn_probability', 'predicted_churn', 'actual_churn', 'tenure', 'MonthlyCharges', 'tenure_group', 'support_score', 'risk_tier', 'month_cohort', 'month_cohort_num', 'annual_revenue_at_risk', 'prob_bucket', 'tenure_group_label', 'risk_score', 'prediction_correct', 'prediction_label']
